[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vkoul/ab-test-practice/blob/main/study_notes/week02_study_validity_threats.ipynb)

> **Run this notebook interactively:** Click the badge above to open in Google Colab.

# Week 2: Test Validity Threats in A/B Testing

**Course:** A/B Testing in Practice ()  
**Company Context:** FamilyNest - Airbnb for families with young kids  
**Role:** Product Data Scientist  

---

## Learning Objectives

By the end of this week, you should be able to:
- Identify and diagnose common threats to A/B test validity
- Detect assignment imbalance using chi-squared tests
- Recognize instrumentation errors and spillover effects
- Distinguish novelty effects from genuine improvements
- Perform dimensional analysis to uncover heterogeneous treatment effects
- Apply a structured workflow for analyzing any A/B test

---

## 1. Test Validity Threats Overview

When we run an A/B test at FamilyNest, we want to make a **causal claim**: "This change caused an improvement in bookings." But several things can go wrong that undermine our ability to make that claim.

### Internal Validity

> **Can we trust the causal claim?**

Internal validity is about whether the observed difference between control and treatment is actually caused by the treatment, or if something else (a confound) could explain it.

**Threats to internal validity:**
- Assignment imbalance (SRM)
- Instrumentation errors (wrong users triggered)
- Interference / spillover between groups
- Novelty / primacy effects

### External Validity

> **Can we generalize the results?**

Even if the causal claim is correct *within the test*, can we expect the same effect when we launch to all users?

**Threats to external validity:**
- Test ran during an unusual period (e.g., school holidays when FamilyNest traffic spikes)
- Test population is not representative of the full user base
- Novelty effect: the effect won't persist at steady state
- Interaction with other running experiments

---

## 2. Assignment Imbalance (Sample Ratio Mismatch - SRM)

### What Is It?

When the ratio of users in control vs. treatment doesn't match what was intended.

**Example at FamilyNest:**  
We set up a 50/50 split for a new search filter experiment, but we observe:
- Control: 52,340 users (52.3%)
- Treatment: 47,660 users (47.7%)

That's not what we expected. Is this just random chance, or is something broken?

### Why It Matters

SRM indicates either:
1. **A bug in the randomization system** (e.g., hash function not distributing evenly)
2. **A systematic bias** (e.g., the treatment causes crashes, so those users never get logged)
3. **Differential attrition** (e.g., treatment users leave the platform at higher rates)

If there's SRM, the groups are no longer comparable, and the test results **cannot be trusted**.

### How to Detect It: Chi-Squared Test

We use a chi-squared test to determine whether the observed split is statistically different from the expected split.

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency

def ab_test_chi2(data_column):
    """
    Test whether the assignment ratio matches the expected 50/50 split.
    
    Parameters
    ----------
    data_column : pd.Series
        A column with group labels (e.g., 'control', 'treatment')
    
    Returns
    -------
    p_value : float
    total_count : int
    observed_counts : pd.Series
    """
    # counts observed
    observed_counts = data_column.value_counts()
    total_count = observed_counts.sum()
    # expected counts (50/50 split)
    expected_counts = [total_count / 2, total_count / 2]
    # chi-squared test
    chi2, p_value, dof, expected = chi2_contingency(
        [[observed_counts.iloc[0], observed_counts.iloc[1]],
         [expected_counts[0], expected_counts[1]]]
    )
    return p_value, total_count, observed_counts

In [ ]:
# Simulate FamilyNest experiment assignment data
np.random.seed(42)

# Scenario 1: Balanced assignment (no SRM)
n_users = 100000
balanced_assignment = pd.Series(
    np.random.choice(['control', 'treatment'], size=n_users, p=[0.50, 0.50])
)

p_val, total, counts = ab_test_chi2(balanced_assignment)
print("=== Scenario 1: Balanced Assignment ===")
print(f"Observed counts:\n{counts}")
print(f"Total users: {total}")
print(f"P-value: {p_val:.4f}")
print(f"SRM detected: {'YES - INVESTIGATE!' if p_val < 0.05 else 'No (assignment looks fine)'}")

In [ ]:
# Scenario 2: Imbalanced assignment (SRM present)
# Simulating a bug where treatment users on iOS are being dropped
imbalanced_assignment = pd.Series(
    np.random.choice(['control', 'treatment'], size=n_users, p=[0.53, 0.47])
)

p_val, total, counts = ab_test_chi2(imbalanced_assignment)
print("=== Scenario 2: Imbalanced Assignment (SRM) ===")
print(f"Observed counts:\n{counts}")
print(f"Total users: {total}")
print(f"P-value: {p_val:.6f}")
print(f"SRM detected: {'YES - INVESTIGATE!' if p_val < 0.05 else 'No (assignment looks fine)'}")

### Decision Rule

| P-value | Interpretation | Action |
|---------|---------------|--------|
| >= 0.05 | No evidence of SRM | Proceed with analysis |
| < 0.05 | SRM detected | **Stop. Investigate before trusting results.** |

### Also Check Balance Across Segments

Even if the overall split is 50/50, there could be imbalance within segments. Always check:
- By **continent** (e.g., are European users disproportionately in treatment?)
- By **device** (e.g., is the iOS app sending users to the wrong group?)
- By **user type** (e.g., new vs. returning families)

In [ ]:
# Check SRM across segments
def check_srm_by_segment(df, group_col, segment_col):
    """
    Check for sample ratio mismatch within each segment.
    
    Parameters
    ----------
    df : pd.DataFrame
    group_col : str - column with 'control'/'treatment'
    segment_col : str - column to segment by (e.g., 'continent', 'device')
    """
    print(f"\n{'='*60}")
    print(f"SRM Check by Segment: {segment_col}")
    print(f"{'='*60}")
    
    for segment_value in df[segment_col].unique():
        segment_data = df[df[segment_col] == segment_value][group_col]
        p_val, total, counts = ab_test_chi2(segment_data)
        flag = " *** SRM DETECTED ***" if p_val < 0.05 else ""
        print(f"  {segment_value:20s} | n={total:6d} | "
              f"control={counts.get('control', 0):5d} | "
              f"treatment={counts.get('treatment', 0):5d} | "
              f"p={p_val:.4f}{flag}")

# Create sample data with a segment-level SRM issue
np.random.seed(123)
n = 50000
continents = np.random.choice(['North America', 'Europe', 'Asia', 'Africa'], 
                               size=n, p=[0.4, 0.35, 0.2, 0.05])

# Introduce SRM only for Africa (simulating a targeting bug)
groups = []
for c in continents:
    if c == 'Africa':
        groups.append(np.random.choice(['control', 'treatment'], p=[0.7, 0.3]))
    else:
        groups.append(np.random.choice(['control', 'treatment'], p=[0.5, 0.5]))

df_srm = pd.DataFrame({
    'group': groups,
    'continent': continents
})

check_srm_by_segment(df_srm, 'group', 'continent')

---

## 3. Instrumentation Errors

### The Problem: Wrong Users in the Experiment

**Triggering** means: only users who actually **see** the change should be counted in the experiment.

### FamilyNest Example

Suppose we're testing a new map-based search experience that only works for listings in Africa. If we include *all* FamilyNest users in the test (including those searching in Europe or North America), we'll dilute the effect because most users never encounter the change.

**Correct approach:** Only include users who searched for listings in Africa.

### Intent-to-Treat vs. As-Treated Analysis

| Approach | Who's Included | Pros | Cons |
|----------|---------------|------|------|
| **Intent-to-Treat (ITT)** | Everyone randomized, regardless of whether they saw the change | Preserves randomization, no selection bias | Diluted effect (includes users who never triggered) |
| **As-Treated** | Only users who actually experienced the change | Larger effect size | Selection bias (users who triggered may differ systematically) |

**Best practice at FamilyNest:**
- Use ITT as the primary analysis (conservative, unbiased)
- Use as-treated as a secondary/exploratory analysis
- Always verify that targeting is working correctly by checking segments

### How to Check for Instrumentation Errors

1. **Segment by relevant dimensions:** If the feature is only for Africa, slice by continent. Treatment effect should only appear in Africa.
2. **Check trigger rates:** What % of treatment users actually saw the change? If it's very low, targeting may be broken.
3. **Verify logging:** Are events being recorded correctly for both groups?

In [ ]:
# Demonstrating an instrumentation error
# Feature is supposed to only affect Africa users, but we included everyone

np.random.seed(42)
n = 10000

# Simulate user data
df_instr = pd.DataFrame({
    'user_id': range(n),
    'continent': np.random.choice(
        ['North America', 'Europe', 'Asia', 'Africa'], 
        size=n, p=[0.4, 0.35, 0.2, 0.05]
    ),
    'group': np.random.choice(['control', 'treatment'], size=n)
})

# The treatment only works for Africa users:
# - Africa users in treatment: +15% booking rate
# - Everyone else: no difference
base_booking_rate = 0.10
df_instr['booked'] = np.random.binomial(
    1, 
    base_booking_rate + 0.015 * (
        (df_instr['group'] == 'treatment') & (df_instr['continent'] == 'Africa')
    ).astype(int)
)

# Overall analysis (WRONG - includes non-triggered users)
overall_control = df_instr[df_instr['group'] == 'control']['booked'].mean()
overall_treatment = df_instr[df_instr['group'] == 'treatment']['booked'].mean()

print("=== Incorrect: Overall Analysis (includes non-triggered users) ===")
print(f"Control booking rate: {overall_control:.4f}")
print(f"Treatment booking rate: {overall_treatment:.4f}")
print(f"Lift: {(overall_treatment - overall_control) / overall_control * 100:.2f}%")
print("\nConclusion: Looks like no effect... but that's misleading!")

print("\n")

# Correct analysis (only Africa users)
africa_df = df_instr[df_instr['continent'] == 'Africa']
africa_control = africa_df[africa_df['group'] == 'control']['booked'].mean()
africa_treatment = africa_df[africa_df['group'] == 'treatment']['booked'].mean()

print("=== Correct: Africa-Only Analysis (triggered users) ===")
print(f"Control booking rate: {africa_control:.4f}")
print(f"Treatment booking rate: {africa_treatment:.4f}")
print(f"Lift: {(africa_treatment - africa_control) / africa_control * 100:.2f}%")
print(f"Africa users: {len(africa_df)} out of {n} total ({len(africa_df)/n*100:.1f}%)")
print("\nConclusion: The treatment works for the targeted population!")

---

## 4. Interference / Spillover Effects

### The Problem

A/B tests assume that one user's assignment doesn't affect another user's outcome. This assumption is called **SUTVA** (Stable Unit Treatment Value Assumption). When it's violated, we have **interference** or **spillover**.

### Why This Matters at FamilyNest (Marketplace Effects)

FamilyNest is a **two-sided marketplace** (families searching + hosts listing). Spillover is very real:

**Scenario:** We test a new "Instant Book" feature that makes booking faster.
- Treatment users book properties faster
- Those properties become unavailable for control users
- Control users see fewer options, reducing their booking rate
- **Result:** We overestimate the treatment effect because control is artificially depressed

**Other examples:**
- A pricing algorithm change that affects supply availability
- A host messaging feature that changes response rates for all guests
- A referral program where treatment users invite control users

### Mitigation Strategies

| Strategy | How It Works | When to Use |
|----------|-------------|-------------|
| **Cluster Randomization** | Randomize by clusters (e.g., all users in a city) | When interactions are within clusters |
| **Geo-Based Experiments** | Randomize entire geographic regions | Marketplace/supply experiments |
| **Time-Based Experiments** | Alternate treatment on/off over time | When geo isn't feasible |
| **Network Isolation** | Identify connected components and randomize at that level | Social features |

### FamilyNest Approach

For supply-side experiments (anything affecting listing availability), we use **geo-based randomization**:
- Assign entire cities or regions to control/treatment
- All users within a region get the same experience
- Eliminates within-region spillover (but reduces sample size)

In [ ]:
# Illustration: How spillover biases results

np.random.seed(42)

# Simulate a marketplace with limited supply
n_users = 1000
n_listings = 100  # limited supply!

# True treatment effect: +5% booking probability
true_effect = 0.05
base_rate = 0.10

# Without spillover (ideal world)
control_rate_no_spillover = base_rate
treatment_rate_no_spillover = base_rate + true_effect

# With spillover (treatment takes supply away from control)
# Treatment books more -> fewer listings for control
spillover_penalty = 0.02  # control rate drops because supply is taken
control_rate_with_spillover = base_rate - spillover_penalty
treatment_rate_with_spillover = base_rate + true_effect

print("=== Impact of Spillover on Estimated Treatment Effect ===")
print(f"\nTrue treatment effect: {true_effect*100:.1f}% absolute lift")
print(f"\nWithout spillover:")
print(f"  Control rate: {control_rate_no_spillover:.2%}")
print(f"  Treatment rate: {treatment_rate_no_spillover:.2%}")
print(f"  Measured effect: {(treatment_rate_no_spillover - control_rate_no_spillover)*100:.1f}%")
print(f"\nWith spillover (treatment takes supply from control):")
print(f"  Control rate: {control_rate_with_spillover:.2%} (depressed by spillover)")
print(f"  Treatment rate: {treatment_rate_with_spillover:.2%}")
print(f"  Measured effect: {(treatment_rate_with_spillover - control_rate_with_spillover)*100:.1f}%")
print(f"  BIAS: +{(treatment_rate_with_spillover - control_rate_with_spillover - true_effect)*100:.1f}% overestimate!")
print(f"\nWe'd think the feature is {(treatment_rate_with_spillover - control_rate_with_spillover)/true_effect:.0%} "
      f"as effective as it truly is.")

---

## 5. Novelty / Primacy Effects

### Novelty Effect

Users are initially **excited** about something new, leading to inflated engagement that fades over time.

**FamilyNest example:** We launch a new "Family Activity Planner" widget on the listing page. In the first week, everyone clicks on it out of curiosity. By week 3, usage drops to a much lower steady state.

**Symptoms:**
- Early results look "too good to be true"
- Effect size decreases over time
- New users (who've never seen the old version) show a smaller effect

### Primacy Effect

Users **resist change** initially because they're used to the old experience, but eventually adapt.

**FamilyNest example:** We move the "Book Now" button from the top of the page to a sticky footer. Initially, bookings drop because users can't find the button. After a week, users adapt and the new position actually performs better.

**Symptoms:**
- Early results look bad, then improve
- Users with longer exposure show better performance

### How to Detect

**Plot results over time by cohort:**

In [ ]:
import matplotlib.pyplot as plt

# Simulate novelty effect at FamilyNest
np.random.seed(42)

days = np.arange(1, 22)  # 3 weeks of experiment

# True steady-state lift is 3%, but novelty adds early boost
true_lift = 0.03
novelty_boost = 0.07 * np.exp(-0.3 * (days - 1))  # decays exponentially

observed_lift = true_lift + novelty_boost
noise = np.random.normal(0, 0.005, len(days))

fig, ax = plt.subplots(1, 1, figsize=(10, 5))
ax.plot(days, (observed_lift + noise) * 100, 'b-o', markersize=4, label='Observed daily lift')
ax.axhline(y=true_lift * 100, color='g', linestyle='--', label=f'True steady-state lift ({true_lift*100:.0f}%)')
ax.axhline(y=0, color='gray', linestyle='-', alpha=0.3)
ax.fill_between(days, 0, (observed_lift + noise) * 100, alpha=0.1, color='blue')

ax.set_xlabel('Day of Experiment')
ax.set_ylabel('Treatment Lift (%)')
ax.set_title('FamilyNest: Novelty Effect in "Family Activity Planner" Test')
ax.legend()
ax.set_ylim(-1, 12)
ax.grid(True, alpha=0.3)

# Annotate
ax.annotate('Novelty spike!\nDon\'t ship based on this.', 
            xy=(2, (observed_lift[1] + noise[1]) * 100),
            xytext=(5, 9),
            arrowprops=dict(arrowstyle='->', color='red'),
            fontsize=10, color='red')

plt.tight_layout()
plt.show()

print("\nKey insight: If we stopped the test on Day 3, we'd think the lift was")
print(f"{observed_lift[2]*100:.1f}%, but the true steady-state lift is only {true_lift*100:.1f}%.")
print("\nRule of thumb: If early results look too good to be true, they probably are.")

In [ ]:
# Simulate primacy effect
np.random.seed(123)

days = np.arange(1, 22)

# True steady-state lift is +2%, but users resist initially
true_lift_primacy = 0.02
primacy_penalty = -0.04 * np.exp(-0.25 * (days - 1))  # resistance decays

observed_lift_primacy = true_lift_primacy + primacy_penalty
noise = np.random.normal(0, 0.004, len(days))

fig, ax = plt.subplots(1, 1, figsize=(10, 5))
ax.plot(days, (observed_lift_primacy + noise) * 100, 'r-o', markersize=4, label='Observed daily lift')
ax.axhline(y=true_lift_primacy * 100, color='g', linestyle='--', label=f'True steady-state lift ({true_lift_primacy*100:.0f}%)')
ax.axhline(y=0, color='gray', linestyle='-', alpha=0.3)

ax.set_xlabel('Day of Experiment')
ax.set_ylabel('Treatment Lift (%)')
ax.set_title('FamilyNest: Primacy Effect in "Moved Book Button" Test')
ax.legend()
ax.set_ylim(-4, 4)
ax.grid(True, alpha=0.3)

ax.annotate('Users can\'t find the button!\n(They\'ll adapt)', 
            xy=(2, (observed_lift_primacy[1] + noise[1]) * 100),
            xytext=(6, -3),
            arrowprops=dict(arrowstyle='->', color='orange'),
            fontsize=10, color='orange')

plt.tight_layout()
plt.show()

print("\nKey insight: If we stopped on Day 3, we'd kill the experiment (negative lift).")
print("But users adapt, and the true steady-state effect is positive.")
print("\nBest practice: Run the test long enough for the effect to stabilize.")

---

## 6. Dimensional Analysis (Segment Breakdowns)

### Why Segment?

The overall average treatment effect can be **misleading** if the effect is concentrated in one segment.

**FamilyNest example:** A new pricing display shows +2% overall booking lift. But when we break down:
- North America: +5% (most of our users)
- Europe: +1% 
- Asia: -3% (the format is confusing for local currencies)

The overall +2% hides a real problem in Asia. We should fix the Asia experience before launching.

### Key Dimensions to Check

At FamilyNest, always break down by:
- **Continent** - different user behaviors and expectations
- **Device** (mobile/desktop/tablet) - different UI experiences
- **Booked previously** (yes/no) - new vs. experienced users
- **Family size** - our key differentiator

### Custom Results Function

In [ ]:
from scipy.stats import ttest_ind

def custom_calculate_results(df, metric_col, group_col='group', 
                              dimensions=None, alpha=0.05):
    """
    Calculate A/B test results overall and broken down by dimensions.
    
    Parameters
    ----------
    df : pd.DataFrame
        Experiment data
    metric_col : str
        The target metric column
    group_col : str
        Column indicating 'control' or 'treatment'
    dimensions : list of str, optional
        Columns to break down by
    alpha : float
        Significance threshold
    
    Returns
    -------
    results : dict with overall and per-dimension results
    """
    results = {}
    
    # Overall results
    control = df[df[group_col] == 'control'][metric_col]
    treatment = df[df[group_col] == 'treatment'][metric_col]
    
    t_stat, p_value = ttest_ind(treatment, control)
    
    results['overall'] = {
        'control_mean': control.mean(),
        'treatment_mean': treatment.mean(),
        'lift_pct': (treatment.mean() - control.mean()) / control.mean() * 100,
        'p_value': p_value,
        'significant': p_value < alpha,
        'n_control': len(control),
        'n_treatment': len(treatment)
    }
    
    # Dimensional breakdowns
    if dimensions:
        for dim in dimensions:
            results[dim] = {}
            for segment_value in df[dim].unique():
                segment_df = df[df[dim] == segment_value]
                seg_control = segment_df[segment_df[group_col] == 'control'][metric_col]
                seg_treatment = segment_df[segment_df[group_col] == 'treatment'][metric_col]
                
                if len(seg_control) > 1 and len(seg_treatment) > 1:
                    t_stat, seg_p = ttest_ind(seg_treatment, seg_control)
                    results[dim][segment_value] = {
                        'control_mean': seg_control.mean(),
                        'treatment_mean': seg_treatment.mean(),
                        'lift_pct': (seg_treatment.mean() - seg_control.mean()) / seg_control.mean() * 100 if seg_control.mean() > 0 else 0,
                        'p_value': seg_p,
                        'significant': seg_p < alpha,
                        'n_control': len(seg_control),
                        'n_treatment': len(seg_treatment)
                    }
    
    return results


def print_results(results):
    """Pretty-print the results dictionary."""
    print("\n" + "="*70)
    print("OVERALL RESULTS")
    print("="*70)
    r = results['overall']
    sig_marker = "***" if r['significant'] else ""
    print(f"  Control mean:   {r['control_mean']:.4f} (n={r['n_control']})")
    print(f"  Treatment mean: {r['treatment_mean']:.4f} (n={r['n_treatment']})")
    print(f"  Lift:           {r['lift_pct']:+.2f}%")
    print(f"  P-value:        {r['p_value']:.4f} {sig_marker}")
    print(f"  Significant:    {r['significant']}")
    
    for key in results:
        if key == 'overall':
            continue
        print(f"\n{'='*70}")
        print(f"BREAKDOWN BY: {key.upper()}")
        print(f"{'='*70}")
        for segment, seg_r in results[key].items():
            sig_marker = "***" if seg_r['significant'] else ""
            print(f"  {segment:20s} | lift: {seg_r['lift_pct']:+6.2f}% | "
                  f"p={seg_r['p_value']:.4f} {sig_marker} | "
                  f"n={seg_r['n_control']+seg_r['n_treatment']}")

In [ ]:
# Create a realistic FamilyNest dataset with heterogeneous treatment effects
np.random.seed(42)
n = 20000

df_exp = pd.DataFrame({
    'user_id': range(n),
    'group': np.random.choice(['control', 'treatment'], size=n),
    'continent': np.random.choice(
        ['North America', 'Europe', 'Asia', 'Africa'],
        size=n, p=[0.4, 0.35, 0.2, 0.05]
    ),
    'device': np.random.choice(['mobile', 'desktop', 'tablet'], size=n, p=[0.55, 0.35, 0.10]),
    'booked_previously': np.random.choice(['yes', 'no'], size=n, p=[0.3, 0.7])
})

# Generate booking outcome with heterogeneous treatment effects
base_rate = 0.10
treatment_effects = {
    'North America': 0.03,
    'Europe': 0.01,
    'Asia': -0.02,  # Negative effect in Asia!
    'Africa': 0.04
}

booking_probs = []
for _, row in df_exp.iterrows():
    prob = base_rate
    if row['group'] == 'treatment':
        prob += treatment_effects[row['continent']]
    # Returning users book more
    if row['booked_previously'] == 'yes':
        prob += 0.05
    booking_probs.append(max(0, min(1, prob)))

df_exp['booked'] = np.random.binomial(1, booking_probs)

# Run the analysis
results = custom_calculate_results(
    df_exp, 
    metric_col='booked',
    dimensions=['continent', 'device', 'booked_previously']
)

print_results(results)

### Caveat: Multiple Comparisons / False Discovery Rate

When you test many segments, you increase the chance of a **false positive**.

If you test 20 segments at alpha=0.05, you expect ~1 false positive by chance alone.

**Mitigation strategies:**
- **Bonferroni correction:** Divide alpha by the number of comparisons (conservative)
- **Benjamini-Hochberg:** Controls the false discovery rate (less conservative)
- **Pre-registration:** Decide which segments to analyze BEFORE looking at results
- **Judgment:** Use segment analysis for hypothesis generation, not confirmation

**Rule of thumb at FamilyNest:**
- Pre-register 2-3 key dimensions (continent, device, user type)
- If a segment shows a surprising result, replicate in a follow-up test
- Don't cherry-pick segments to make results look good

---

## 7. Practical Workflow for Analyzing an A/B Test

### Step-by-Step Checklist

Use this checklist every time you analyze an A/B test at FamilyNest:

```
[ ] Step 1: Check for assignment imbalance (chi-squared test)
    - Overall SRM test
    - SRM by key segments
    - If SRM detected: STOP, investigate the bug

[ ] Step 2: Verify correct targeting/triggering
    - Are the right users in the experiment?
    - What % of treatment users saw the change?
    - Segment by targeting criteria to confirm

[ ] Step 3: Analyze target metric
    - Is the result statistically significant?
    - What direction? What magnitude?
    - Is the effect practically meaningful?

[ ] Step 4: Check guardrail metrics
    - Are there negative side effects?
    - Revenue, engagement, latency, error rates

[ ] Step 5: Break down by key dimensions
    - Continent, device, user type
    - Look for heterogeneous effects
    - Check for novelty/primacy over time

[ ] Step 6: Make launch recommendation with reasoning
    - Ship / Don't ship / Iterate / Extend test
    - Document the evidence
```

In [ ]:
def analyze_ab_test(df, group_col, metric_col, dimensions, 
                     guardrail_metrics=None, expected_ratio=0.5):
    """
    Complete A/B test analysis workflow for FamilyNest.
    
    Parameters
    ----------
    df : pd.DataFrame
    group_col : str - column with 'control'/'treatment'
    metric_col : str - primary target metric
    dimensions : list of str - columns to segment by
    guardrail_metrics : list of str, optional - additional metrics to check
    expected_ratio : float - expected fraction in control (default 0.5)
    """
    print("#" * 70)
    print("#  FamilyNest A/B Test Analysis Report")
    print("#" * 70)
    
    # =============================================
    # STEP 1: Assignment Imbalance Check
    # =============================================
    print("\n" + "="*70)
    print("STEP 1: ASSIGNMENT IMBALANCE (SRM) CHECK")
    print("="*70)
    
    p_val, total, counts = ab_test_chi2(df[group_col])
    srm_detected = p_val < 0.05
    
    print(f"  Total users: {total:,}")
    print(f"  Control: {counts.get('control', 0):,} ({counts.get('control', 0)/total*100:.1f}%)")
    print(f"  Treatment: {counts.get('treatment', 0):,} ({counts.get('treatment', 0)/total*100:.1f}%)")
    print(f"  Chi-squared p-value: {p_val:.4f}")
    
    if srm_detected:
        print("\n  *** WARNING: SRM DETECTED! ***")
        print("  Investigate before proceeding. Results may not be trustworthy.")
    else:
        print("\n  PASS: No evidence of assignment imbalance.")
    
    # =============================================
    # STEP 2: Targeting Verification
    # =============================================
    print("\n" + "="*70)
    print("STEP 2: TARGETING VERIFICATION")
    print("="*70)
    print("  (Manual check - verify with engineering that targeting is correct)")
    print(f"  Unique users in experiment: {df['user_id'].nunique() if 'user_id' in df.columns else len(df):,}")
    
    # =============================================
    # STEP 3: Target Metric Analysis
    # =============================================
    print("\n" + "="*70)
    print(f"STEP 3: TARGET METRIC ANALYSIS ({metric_col})")
    print("="*70)
    
    control_metric = df[df[group_col] == 'control'][metric_col]
    treatment_metric = df[df[group_col] == 'treatment'][metric_col]
    t_stat, p_value = ttest_ind(treatment_metric, control_metric)
    
    lift = (treatment_metric.mean() - control_metric.mean()) / control_metric.mean() * 100
    
    print(f"  Control mean: {control_metric.mean():.4f}")
    print(f"  Treatment mean: {treatment_metric.mean():.4f}")
    print(f"  Relative lift: {lift:+.2f}%")
    print(f"  P-value: {p_value:.4f}")
    print(f"  Statistically significant: {p_value < 0.05}")
    
    if p_value < 0.05:
        direction = "POSITIVE" if lift > 0 else "NEGATIVE"
        print(f"\n  Result: {direction} and SIGNIFICANT")
    else:
        print(f"\n  Result: NOT SIGNIFICANT (cannot reject null hypothesis)")
    
    # =============================================
    # STEP 4: Guardrail Metrics
    # =============================================
    print("\n" + "="*70)
    print("STEP 4: GUARDRAIL METRICS")
    print("="*70)
    
    if guardrail_metrics:
        for gm in guardrail_metrics:
            if gm in df.columns:
                g_control = df[df[group_col] == 'control'][gm]
                g_treatment = df[df[group_col] == 'treatment'][gm]
                g_t, g_p = ttest_ind(g_treatment, g_control)
                g_lift = (g_treatment.mean() - g_control.mean()) / g_control.mean() * 100 if g_control.mean() > 0 else 0
                flag = "DEGRADED" if (g_p < 0.05 and g_lift < 0) else "OK"
                print(f"  {gm:25s} | lift: {g_lift:+.2f}% | p={g_p:.4f} | {flag}")
    else:
        print("  No guardrail metrics specified.")
    
    # =============================================
    # STEP 5: Dimensional Breakdown
    # =============================================
    print("\n" + "="*70)
    print("STEP 5: DIMENSIONAL BREAKDOWN")
    print("="*70)
    
    results = custom_calculate_results(df, metric_col, group_col, dimensions)
    for dim in dimensions:
        if dim in results:
            print(f"\n  --- {dim.upper()} ---")
            for segment, seg_r in results[dim].items():
                sig_marker = "***" if seg_r['significant'] else ""
                print(f"    {segment:20s} | lift: {seg_r['lift_pct']:+6.2f}% | "
                      f"p={seg_r['p_value']:.4f} {sig_marker}")
    
    # =============================================
    # STEP 6: Recommendation
    # =============================================
    print("\n" + "="*70)
    print("STEP 6: LAUNCH RECOMMENDATION")
    print("="*70)
    
    # Simple decision logic
    if srm_detected:
        rec = "DO NOT SHIP - Fix SRM issue first"
    elif p_value >= 0.05:
        rec = "NEUTRAL - No significant effect detected. Consider extending test or iterating."
    elif lift > 0:
        # Check if any segment is significantly negative
        any_negative_segment = False
        if dimensions:
            for dim in dimensions:
                if dim in results:
                    for seg, seg_r in results[dim].items():
                        if seg_r['significant'] and seg_r['lift_pct'] < -2:
                            any_negative_segment = True
        if any_negative_segment:
            rec = "ITERATE - Positive overall but some segments degraded. Fix before full launch."
        else:
            rec = "SHIP - Positive, significant, no segment concerns."
    else:
        rec = "DO NOT SHIP - Significant negative effect."
    
    print(f"\n  >>> {rec}")
    print("\n" + "#" * 70)
    
    return results

In [ ]:
# Run the full workflow on our FamilyNest experiment

# Add a guardrail metric (pages viewed per session)
df_exp['pages_viewed'] = np.random.poisson(5, n) + \
    (df_exp['group'] == 'treatment').astype(int) * np.random.choice([0, 1], n, p=[0.8, 0.2])

# Run complete analysis
full_results = analyze_ab_test(
    df=df_exp,
    group_col='group',
    metric_col='booked',
    dimensions=['continent', 'device', 'booked_previously'],
    guardrail_metrics=['pages_viewed']
)

---

## Summary: Key Takeaways

| Threat | Detection Method | Action |
|--------|-----------------|--------|
| **SRM** | Chi-squared test on assignment counts | Stop analysis, find the bug |
| **Instrumentation** | Segment by targeting criteria | Fix targeting, re-run test |
| **Spillover** | Domain knowledge + cluster experiments | Use geo/cluster randomization |
| **Novelty/Primacy** | Plot lift over time by cohort | Run test longer, use steady-state estimate |
| **HTE** | Dimensional breakdown | Document, consider segment-specific launches |

### Golden Rules for FamilyNest Data Scientists

1. **Always check SRM first.** If the assignment is broken, nothing else matters.
2. **Verify targeting.** Are the right users in the experiment?
3. **Don't peek too early.** Novelty effects can fool you.
4. **Think about spillover** for marketplace features.
5. **Segment responsibly.** Pre-register dimensions, beware of multiple comparisons.
6. **Document your reasoning.** Every ship/no-ship decision should have a written rationale.

---

*Week 2 Study Guide - FamilyNest A/B Testing Course*